In [1]:
%cd /home/kawamanmi/Projects/PyBaMM
import pybamm
import numpy as np
import matplotlib.pyplot as plt


# Stadardised Formation Cycling
# exp = pybamm.Experiment( [("Charge at C/20 until 4.2 V",
#                            "Hold at 4.2 V for 1 hours", 
#                            "Hold at 4.2 V until 2.5 mA", 
#                            "Discharge at C/20 until 2.5 V",
#                            "Charge at C/20 until 3.8 V",
#                            "Hold at 4.8 V for 1 hours"
#                            )])



no_cycles = 2
exp = pybamm.Experiment(
    [("Charge at C/20 until 4.2 V",
      "Rest for 5 hours",
      "Discharge at C/20 until 2.5 V",
      "Rest for 5 hours",
      )]* no_cycles )

param = pybamm.ParameterValues("Chen2020")
param['Initial inner SEI thickness [m]'] = 5e-12
param['Initial outer SEI thickness [m]'] = 5e-12

pybamm.settings.max_y_value = 1000000000
solver = pybamm.CasadiSolver(mode="safe")



/home/kawamanmi/Projects/PyBaMM


In [2]:
# param

In [3]:
model = pybamm.lithium_ion.SPMe()
sim = pybamm.Simulation(model, parameter_values=param,
                        experiment=exp, solver=solver)
sol = sim.solve(initial_soc=0)

In [4]:
pybamm.dynamic_plot(sol, output_variables=["Electrolyte potential [V]",
                                           "Discharge capacity [A.h]",
                                           "Terminal voltage [V]", "Power [W]",
                                           "X-averaged negative particle surface concentration [mol.m-3]",
                                           'X-averaged negative electrode potential [V]',
                                           'Total current density [A.m-2]',
                                           'Current [A]',
                                           ],
                    variable_limits='tight')

interactive(children=(FloatSlider(value=0.0, description='t', max=101.24510677126348, step=1.0124510677126348)…

In [5]:
# Finding overpotntial (eta_sei)
delta_phi = sol['X-averaged negative electrode surface potential difference [V]'].entries
U_sei = 0.4
J_app = sol['Total current density [A.m-2]'].entries
L_sei = 2*sol['X-averaged negative total SEI thickness [m]'].entries
R_sei = 200000.0
eta_SEI = delta_phi - U_sei - J_app * L_sei * R_sei
F = 96485
R = 8.314
T = 298.15
alpah = 0.5
# phi_n = sol['X-averaged negative electrode potential [V]'].entries
# phi_e = sol['X-averaged electrolyte potential [V]'].entries
#  eta_SEI = delta_phi - phase_param.U_sei - j * L_sei * R_sei
# 'Inner SEI open-circuit potential [V]': 0.1,
# print(phi_n_phi_e-phi_n+phi_e)
# plt.plot(sol['Time [h]'].entries, np.exp(-alpah * F/(R * T) * eta_SEI))
# plt.show()
# print(np.min(np.exp(-alpah * F/(R * T) * eta_SEI)))
# print(np.max(np.exp(-alpah * F/(R * T) * eta_SEI)))


# D_sol = np.linspace(1e-23, 1e-13, 6)
# C_sol = np.linspace(2000, 5000, 6)
# k_sei_0 = np.linspace(1e-19, 1e-5, 6)
# DD_max = []
# DD_min = []
# RR_max = []
# RR_min = []
# for D in D_sol:
#     for C in C_sol:
#         for k in k_sei_0:
#             DD = J_app * 1e-9 / (F*D*C)
#             RR = J_app / (F*k*C * np.exp(-alpah * F/(R * T) * eta_SEI))
#             DD_max.append(np.max(DD))
#             DD_min.append(np.min(DD))
#             RR_max.append(np.max(RR))
#             RR_min.append(np.min(RR))
# print(DD_max)
# print(DD_min)
# print(RR_max)
# print(RR_min)


# If consider maximum values for all parameters
eta_max = np.max(np.exp(-alpah * F/(R * T) * eta_SEI))
J_app = np.max(sol['Total current density [A.m-2]'].entries)
D = 1e-13
C = 4541
k = 1.3e-5

RR = F*k*C*eta_max
print(RR)
DD = J_app * 1e-12 / (F*D*C)
print(DD)


# If consider minimum values for all parameters
eta_min = np.min(np.exp(-alpah * F/(R * T) * eta_SEI))
J_app = np.abs(np.min(sol['Total current density [A.m-2]'].entries))
D = 1e-23
C = 2636
k = 1.3e-19

RR = F*k*C*eta_min
print(RR)
DD = J_app * 1e-12 / (F*D*C)
print(DD)

2744481.1381374323
5.555949146462755e-08
4.0424866130120023e-17
957.1155187438306


In [6]:
model.variable_names()

['Time [s]',
 'Time [min]',
 'Time [h]',
 'x [m]',
 'x_n [m]',
 'x_s [m]',
 'x_p [m]',
 'r_n [m]',
 'r_p [m]',
 'Current variable [A]',
 'Total current density [A.m-2]',
 'Current [A]',
 'C-rate',
 'Discharge capacity [A.h]',
 'Throughput capacity [A.h]',
 'Discharge energy [W.h]',
 'Throughput energy [W.h]',
 'Porosity',
 'Negative electrode porosity',
 'X-averaged negative electrode porosity',
 'Separator porosity',
 'X-averaged separator porosity',
 'Positive electrode porosity',
 'X-averaged positive electrode porosity',
 'Porosity change',
 'Negative electrode porosity change [s-1]',
 'X-averaged negative electrode porosity change [s-1]',
 'Separator porosity change [s-1]',
 'X-averaged separator porosity change [s-1]',
 'Positive electrode porosity change [s-1]',
 'X-averaged positive electrode porosity change [s-1]',
 'Negative electrode interface utilisation variable',
 'X-averaged negative electrode interface utilisation variable',
 'Negative electrode interface utilisation',


In [7]:
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
import scienceplots
import matplotlib.cm as cm
for index in range(len(sols)):
    plt.plot(sols[index]["Time [h]"].entries,
             sols[index]['X-averaged negative electrode reaction overpotential [V]'].entries, label=c_rates[index])
plt.legend()

NameError: name 'sols' is not defined